In [ ]:
#!pip uninstall -y transformers tokenizers librosa
#!pip install -U transformers accelerate
#!pip install -U librosa soundfile soxr

In [1]:
# Imports
import pandas as pd
from pathlib import Path
import os
import glob
import random
import numpy as np
import re
import time
import torch
from tqdm.auto import tqdm
from google.colab import files

In [2]:
# Import model quicker
#from huggingface_hub import login
#from google.colab import userdata

#login(token=userdata.get(""))

In [3]:
# Setting all seed options for reproducibility
SEED = 42

random.seed(SEED) # Python
np.random.seed(SEED) # NumPy
torch.manual_seed(SEED) # PyTorch

torch.cuda.manual_seed(SEED) # Cuda
torch.cuda.manual_seed_all(SEED) # Cuda

os.environ["PYTHONHASHSEED"] = str(SEED) # More Python


In [4]:
# Importhing the Qwen model --> https://huggingface.co/Qwen/Qwen2-Audio-7B-Instruct

from io import BytesIO
from urllib.request import urlopen
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
model = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct", device_map="auto")

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/638k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

In [5]:
# Import the RAVDESS dataset --> https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
import kagglehub
ravdess = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.


In [6]:
# Mapping the emotion labels
emotions = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

# Use the first 4 actors for evaluation, the remaining 20 actors are for finetuning the model
actors_val = list(range(1,5))
actors_train = list(range(5, 25))

# Searching the specific paths
audio_files = glob.glob(os.path.join(ravdess, "**", "*.wav"), recursive=True)

records = []
seen = set()  # Use set to make sure there are no duplicates

for filepath in audio_files:

    # File name should look like:"03-01-06-01-02-01-12.wav"
    filename = os.path.basename(filepath)
    parts = filename.replace(".wav", "").split("-")

    if len(parts) != 7:
        continue

    # Relevant filename identifiers
    modality = parts[0]
    channel = parts[1]
    emotion_code = parts[2]
    actor = int(parts[6])

    # Modality should be 03 = Audio-only
    # Channel should be 01 = speech
    if modality != "03" or channel != "01":
        continue

    # To make the valid emotions are used
    if emotion_code not in emotions:
        continue

    # To make sure there are no duplicates
    if filename in seen:
        continue
    seen.add(filename)

    records.append({
        "path": filepath,
        "filename": filename,
        "emotion": emotions[emotion_code],
        "emotion_code": emotion_code,
        "actor": actor})

df = pd.DataFrame(records)

df["split"] = df["actor"].apply(
    lambda x: "evaluation"
    if x in actors_val
    else "train")

train_df = df[df["split"] == "train"].reset_index(drop=True)
eval_df = df[df["split"] == "evaluation"].reset_index(drop=True)


In [7]:
# The task given to QWEN
instruction_1 = (
    "Listen carefully to the speaker's voice and identify the emotion"
    "which the speaker expresses through their tone, pitch, energy ect. "
    "Classify the emotion based on the voice, not the meaning of the words. "
    "Choose exactly ONE of these eight categories: "
    "neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
    "Respond with the emotion label only. Do not provide an explanation.")

In [17]:
emotion_labels = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]

def extract_emotion(response):
    response = response.lower().strip()

    for emotion in emotion_labels:
        if re.search(r"\b" + re.escape(emotion) + r"\b", response):
            return emotion
    # Exception because for some reason the model returns "disgusted" instead of "disgust"
    if response == 'disgusted':
      return "disgust"

    return "invalid"

In [18]:
def predict_emotion(audio_path, instruction):
    # Preprocesses the audio so QWEN can understand it
    audio, sr = librosa.load(audio_path, sr=processor.feature_extractor.sampling_rate)

    # Conversation structure of QWEN
    conversation = [
        {"role": "user",
          "content": [
                {"type": "audio"},
                {"type": "text", "text": instruction}
            ]}
    ]

    # Let QWEN generate the speaker's emotion
    text = processor.apply_chat_template(conversation, add_generation_prompt=True,
        tokenize=False)

    inputs = processor(text=text, audio=audio, sampling_rate=sr,
        return_tensors="pt", padding=True)

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=32,
            do_sample=False)

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

    response = processor.batch_decode(generated_ids, skip_special_tokens=True,
        clean_up_tokenization_spaces=False)[0]

    return extract_emotion(response), response

In [20]:
OUTPUT_CSV = "/content/ravdess_baseline_predictions.csv"
# Start a fresh CSV
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

In [21]:
# Evaluating the baseline model +
# Show live progress information to know how far along we are 🫡
results = []
start_time = time.time()

print("Start baseline evaluation")

progress = tqdm(eval_df.iterrows(), total=len(eval_df),
    desc="Evaluating RAVDESS", unit="recording",  dynamic_ncols=True)

for i, (_, sample) in enumerate(progress, start=1):
    recording_start = time.time()

    try:
        predicted_emotion, raw_response = predict_emotion(sample["path"], instruction_1)
        result = {
            "filename": sample["filename"],
            "actor": sample["actor"],
            "actual_emotion": sample["emotion"],
            "predicted_emotion": predicted_emotion}

    except Exception as e:
        print(f"\nError processing {sample['filename']}: {e}")

    results.append(result)

    # Save immediately after each recording, so in case of error not all progress is lost
    pd.DataFrame([result]).to_csv(OUTPUT_CSV, mode="a",
        header=not os.path.exists(OUTPUT_CSV), index=False)

    elapsed = time.time() - start_time
    recording_time = time.time() - recording_start

    progress.set_postfix({
        "Actor": sample["actor"],
        "Actual": sample["emotion"],
        "Predicted": result["predicted_emotion"],
        "Last": f"{recording_time:.1f}s"})

progress.close()

# Download the csv after finishing
files.download("/content/ravdess_baseline_predictions.csv")

Start baseline evaluation


Evaluating RAVDESS:   0%|          | 0/240 [00:00<?, ?recording/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Second use scenario
Asking QWEN without specification to classify the speakers emotion

In [22]:
OUTPUT_CSV = "/content/ravdess_baseline_predictions_2.csv"
# Start a fresh CSV
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

In [23]:
instruction_2 = (
    "Listen carefully to the speaker's voice and identify the emotion. "
    "Choose exactly ONE of these eight categories: "
    "neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
    "Respond with the emotion label only. Do not provide an explanation.")

In [24]:
# Evaluating the baseline model +
# Show live progress information to know how far along we are 🫡
results = []
start_time = time.time()

print("Start baseline evaluation")

progress = tqdm(eval_df.iterrows(), total=len(eval_df),
    desc="Evaluating RAVDESS", unit="recording",  dynamic_ncols=True)

for i, (_, sample) in enumerate(progress, start=1):
    recording_start = time.time()

    try:
        predicted_emotion, raw_response = predict_emotion(sample["path"], instruction_2)
        result = {
            "filename": sample["filename"],
            "actor": sample["actor"],
            "actual_emotion": sample["emotion"],
            "predicted_emotion": predicted_emotion}

    except Exception as e:
        print(f"\nError processing {sample['filename']}: {e}")

    results.append(result)

    # Save immediately after each recording, so in case of error not all progress is lost
    pd.DataFrame([result]).to_csv(OUTPUT_CSV, mode="a",
        header=not os.path.exists(OUTPUT_CSV), index=False)

    elapsed = time.time() - start_time
    recording_time = time.time() - recording_start

    progress.set_postfix({
        "Actor": sample["actor"],
        "Actual": sample["emotion"],
        "Predicted": result["predicted_emotion"],
        "Last": f"{recording_time:.1f}s"})

progress.close()

# Download the csv after finishing
files.download("/content/ravdess_baseline_predictions_2.csv")

Start baseline evaluation


Evaluating RAVDESS:   0%|          | 0/240 [00:00<?, ?recording/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>